# TennisVision - Player Pose Detection Test (GPU)

Test yolo26m-pose player detection + pose estimation on tennis video.

**Before running:** Add your dataset (containing `sample_short.mp4`) via Input on the left panel.

In [ ]:
!pip install ultralytics -q

In [ ]:
import cv2
import time
from ultralytics import YOLO

# ---- Adjust this path to your dataset ----
INPUT_VIDEO = "/kaggle/input/tennisvision-test/sample_short.mp4"
OUTPUT_VIDEO = "/kaggle/working/sample_short_pose_detect.mp4"

model = YOLO("yolo26m-pose.pt")  # auto-downloads ~47MB

cap = cv2.VideoCapture(INPUT_VIDEO)
fps = cap.get(cv2.CAP_PROP_FPS)
w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
print(f"Video: {w}x{h}  fps={fps:.1f}  frames={total}")

writer = cv2.VideoWriter(OUTPUT_VIDEO, cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))

colors = [(0,255,0), (255,0,0), (0,255,255), (255,0,255), (0,165,255), (255,255,0)]
SKELETON = [
    (5,6),(5,7),(7,9),(6,8),(8,10),
    (5,11),(6,12),(11,12),
    (11,13),(13,15),(12,14),(14,16),
]

fi = 0
t0 = time.time()
while True:
    ret, frame = cap.read()
    if not ret:
        break
    results = model.track(frame, persist=True, verbose=False, classes=[0], conf=0.15, imgsz=1280)
    r = results[0]
    if r.boxes is not None and r.boxes.id is not None and r.keypoints is not None:
        boxes = r.boxes.xyxy.cpu().numpy()
        ids = r.boxes.id.cpu().numpy().astype(int)
        confs = r.boxes.conf.cpu().numpy()
        kps = r.keypoints.data.cpu().numpy()
        for i in range(len(boxes)):
            x0, y0, x1, y1 = boxes[i].astype(int)
            tid = ids[i]
            c = colors[abs(tid) % len(colors)]
            cv2.rectangle(frame, (x0,y0), (x1,y1), c, 2)
            cv2.putText(frame, f"id={tid} h={y1-y0} conf={confs[i]:.2f}", (x0,y0-8), cv2.FONT_HERSHEY_SIMPLEX, 0.6, c, 2)
            kp = kps[i]
            for j in range(17):
                if kp[j][2] >= 0.3:
                    cv2.circle(frame, (int(kp[j][0]), int(kp[j][1])), 4, c, -1)
            for j1,j2 in SKELETON:
                if kp[j1][2] >= 0.3 and kp[j2][2] >= 0.3:
                    cv2.line(frame, (int(kp[j1][0]),int(kp[j1][1])), (int(kp[j2][0]),int(kp[j2][1])), c, 2)
    writer.write(frame)
    fi += 1
    if fi % 500 == 0:
        elapsed = time.time() - t0
        print(f"  frame {fi}/{total}  {fi/elapsed:.1f} fps  eta {(total-fi)/(fi/elapsed):.0f}s")

cap.release()
writer.release()
print(f"\nDone: {fi} frames in {time.time()-t0:.1f}s")
print(f"Saved: {OUTPUT_VIDEO}")